# Phase 9: Weather Telemetry and Environmental Context Integration
### AI-Driven Intelligent Greywater Management and Smart Reuse Routing System

---
**Core Safety Axiom:**
> *Weather information provides operational context and does not independently establish water safety. Severe water-quality hazards and anomaly cutoffs must NEVER be overridden by favorable weather.*



## 1. Objective

The primary objective of Phase 9 is to introduce an **environmental-context layer** that evaluates meteorological conditions (rainfall, precipitation probability, ambient temperature, humidity) to arbitrate and refine the *initial* routing decision produced by the supervised ML models (XGBoost) and anomaly detector (Isolation Forest).

### The Multi-Stage Routing Pipeline
```
Digital Twin / Telemetry
          ↓
  Water Quality Analysis
          ↓
Isolation Forest (Safety Filter)
          ↓
  XGBoost Initial Route
          ↓
[WEATHER CONTEXT LAYER]  <--- PHASE 9
          ↓
  Storage / Decay (Phase 10)
          ↓
   Final Smart Route
```



In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import context
from context.config import (
    WEATHER_LATITUDE,
    WEATHER_LONGITUDE,
    WEATHER_CITY,
    HIGH_RAINFALL_THRESHOLD_MM,
    HIGH_PRECIP_PROBABILITY_THRESHOLD,
    FREEZING_TEMP_THRESHOLD_C,
    TORRENTIAL_WASHOUT_RAIN_MM,
    OFFLINE_DEMO_PROFILES
)
from context.weather_provider import (
    OfflineWeatherProvider,
    OpenMeteoWeatherProvider,
    ModularWeatherProvider,
    validate_normalized_weather
)
from context.weather_context import (
    evaluate_weather_context,
    WeatherStatus,
    WeatherAdjustment,
    FinalContextRecommendation
)
from context.weather_cache import WeatherCache

print("Weather Context subsystem successfully initialized.")
print(f"Deployment Location: {WEATHER_CITY} ({WEATHER_LATITUDE}, {WEATHER_LONGITUDE})")



## 2. Weather Provider Architecture

The weather integration is designed as a modular, pluggable abstraction (`BaseWeatherProvider`).
- **`BaseWeatherProvider`**: Interface defining `get_current_weather()`, `get_forecast()`, and `normalize_weather_data()`.
- **`OpenMeteoWeatherProvider`**: Public live weather REST client requiring no private credentials.
- **`OfflineWeatherProvider`**: Deterministic meteorological scenario generator for air-gapped testbeds and offline demonstrations.
- **`ModularWeatherProvider`**: Facade coordinating cache checks, live network calls, and instant fallback to offline safety profiles upon network failure.



In [ ]:
# Inspect available offline scenario profiles
profiles_df = pd.DataFrame(OFFLINE_DEMO_PROFILES).T
profiles_df = profiles_df[["weather_condition", "temperature_C", "humidity_percent", "rainfall_mm", "precipitation_probability", "wind_speed_kmh"]]
profiles_df.index.name = "Profile Name"
profiles_df



## 3. API Configuration & Security

Credentials and network parameters are dynamically resolved through environment variables (`.env`), guaranteeing that no secret keys are hard-coded into repository source files.
- `WEATHER_API_KEY`: Strictly optional for default Open-Meteo provider.
- `WEATHER_LATITUDE` / `WEATHER_LONGITUDE`: Geographic coordinates of the treatment facility.
- `WEATHER_API_TIMEOUT_SEC`: Maximum HTTP timeout window (default 5 sec).
- `WEATHER_CACHE_TTL_SEC`: In-memory caching lifespan (default 900 sec / 15 min).



In [ ]:
print("Configuration Settings:")
print(f"  Provider:                  {context.WeatherConfig.PROVIDER}")
print(f"  Latitude / Longitude:      {context.WeatherConfig.LATITUDE}, {context.WeatherConfig.LONGITUDE}")
print(f"  City Name:                 {context.WeatherConfig.CITY}")
print(f"  API Timeout:               {context.WeatherConfig.API_TIMEOUT_SEC} s")
print(f"  Cache TTL:                 {context.WeatherConfig.CACHE_TTL_SEC} s")
print(f"  API Key configured:        {'YES' if context.WeatherConfig.API_KEY else 'NO (Unneeded for Open-Meteo)'}")



## 4. Canonical Weather Data Structure

To decouple downstream routing logic from idiosyncratic external API JSON structures, all provider outputs are normalized into a standardized internal schema:

| Field Name | Type | Physical Unit | Description |
|:---|:---:|:---:|:---|
| `timestamp` | str | UTC timestamp | Observation timestamp |
| `location` | str | - | Geographic municipality identifier |
| `temperature_C` | float | °C | Ambient 2m dry-bulb temperature |
| `humidity_percent` | float | % | Relative humidity [0, 100] |
| `rainfall_mm` | float | mm | Current / hourly precipitation depth |
| `precipitation_probability` | float | % | Forecast probability of precipitation [0, 100] |
| `weather_condition` | str | - | Descriptive WMO classification |
| `wind_speed_kmh` | float | km/h | 10m surface wind speed |
| `forecast_horizon_hours`| int | hours | Forward-looking forecast window |
| `weather_status` | str | - | Status indicator ('LIVE', 'OFFLINE_FALLBACK', 'UNAVAILABLE') |



In [ ]:
offline_provider = OfflineWeatherProvider(profile_name="dry_clear")
sample_weather = offline_provider.get_current_weather()

for k, v in sample_weather.items():
    print(f"  {k:30s}: {v}")



## 5. Weather Normalization and Bounds Validation

The `validate_normalized_weather()` routine guarantees that all incoming telemetry adheres strictly to physical laws:
1. Precipitation probability must lie strictly within $[0, 100]\%$.
2. Rainfall cannot be negative ($\ge 0.0	ext{ mm}$).
3. Relative humidity must lie within $[0, 100]\%$.
4. Temperature must be numeric.



In [ ]:
# Validate clean record
is_valid = validate_normalized_weather(sample_weather)
print(f"Clean weather record validation result: {is_valid}")

# Test bounds checking on out-of-range probability
try:
    bad_record = sample_weather.copy()
    bad_record["precipitation_probability"] = 150.0  # Invalid
    validate_normalized_weather(bad_record)
except ValueError as e:
    print(f"Correctly caught out-of-bounds error: {e}")



## 6. Offline Mode & Fault-Tolerant Fallback

Industrial water recycling facilities require zero-downtime operation. If network access drops, DNS fails, or public API limits are reached:
1. The system **never crashes** or terminates the pipeline.
2. It returns a structured `weather_status = "UNAVAILABLE"`.
3. It defaults to a safe operational baseline under operator advisory.



In [ ]:
# Instantiate modular provider with offline mode enabled
mod_provider = ModularWeatherProvider(offline_mode=True, offline_profile="heavy_rain")
rain_data = mod_provider.get_weather()
print("Offline Provider Telemetry:")
print(f"  Status:    {rain_data['weather_status']}")
print(f"  Condition: {rain_data['weather_condition']}")
print(f"  Rainfall:  {rain_data['rainfall_mm']} mm")
print(f"  Precip %:  {rain_data['precipitation_probability']} %")



## 7. Weather Context Decision Rules & Thresholds

Operational decision thresholds are grounded in soil mechanics and hydraulic engineering:
- **`HIGH_RAINFALL_THRESHOLD_MM` = 5.0 mm**: Active surface rainfall creates soil saturation, leading to pooling and nutrient washout.
- **`HIGH_PRECIP_PROBABILITY_THRESHOLD` = 50.0%**: High likelihood of imminent rainfall warrants pre-emptive deferral of landscape irrigation.
- **`FREEZING_TEMP_THRESHOLD_C` = 3.0°C**: Ground freezing prevents infiltration and risks burst irrigation pipes.
- **`TORRENTIAL_WASHOUT_RAIN_MM` = 25.0 mm**: Extreme storm events that could wash out outdoor bio-filtration media.



In [ ]:
print("Operational Decision Thresholds:")
print(f"  High Rainfall Threshold:           {HIGH_RAINFALL_THRESHOLD_MM} mm")
print(f"  High Precip Probability Threshold: {HIGH_PRECIP_PROBABILITY_THRESHOLD} %")
print(f"  Freezing Ground Threshold:         {FREEZING_TEMP_THRESHOLD_C} °C")
print(f"  Torrential Washout Threshold:      {TORRENTIAL_WASHOUT_RAIN_MM} mm")



## 8. Irrigation Routing Context

When the ML model's initial routing is **Restricted Irrigation (Class 2)**:
- **Low rain & low probability**: Irrigation proceeds normally (`ALLOW_ROUTE`).
- **Rain $\ge 5$ mm or probability $\ge 50\%$**: Greywater irrigation is suppressed to avoid soil saturation, ponding, and runoff (`STORE_FOR_LATER` / `SUPPRESS_IRRIGATION`).
- **Freezing $< 3^\circ	ext{C}$**: Sub-surface freezing suppresses irrigation (`DEFER_ROUTE` / `DEFER_IRRIGATION`).



In [ ]:
dry_weather = OFFLINE_DEMO_PROFILES["dry_clear"]
rain_weather = OFFLINE_DEMO_PROFILES["heavy_rain"]
cold_weather = OFFLINE_DEMO_PROFILES["freezing_cold"]

print("--- Scenario A: Irrigation in Dry Weather ---")
eval_a = evaluate_weather_context("Restricted Irrigation", dry_weather)
print(f"Recommendation: {eval_a['final_context_recommendation']}")
print(f"Reason:         {eval_a['reason']}")
print()

print("--- Scenario B: Irrigation in Heavy Rain ---")
eval_b = evaluate_weather_context("Restricted Irrigation", rain_weather)
print(f"Recommendation: {eval_b['final_context_recommendation']}")
print(f"Adjustment:     {eval_b['weather_adjustment']}")
print(f"Reason:         {eval_b['reason']}")
print()

print("--- Scenario C: Irrigation in Freezing Weather ---")
eval_c = evaluate_weather_context("Restricted Irrigation", cold_weather)
print(f"Recommendation: {eval_c['final_context_recommendation']}")
print(f"Adjustment:     {eval_c['weather_adjustment']}")
print(f"Reason:         {eval_c['reason']}")



## 9. Indoor Reuse Context (Hydraulic Decoupling)

Unlike landscape irrigation, indoor non-potable reuse (such as toilet flushing and custodial washing) operates within an enclosed, climate-controlled municipal building envelope.
Therefore:
- Ambient precipitation does **not** restrict toilet flushing demand.
- Heavy rain does **not** invalidate an initial `Indoor Reuse` decision.
- The weather context engine maintains `ALLOW_ROUTE` with neutral adjustment.



In [ ]:
indoor_eval = evaluate_weather_context("Indoor Reuse", rain_weather)
print("Indoor Reuse during Torrential Rain:")
print(f"  Weather Status: {indoor_eval['weather_status']}")
print(f"  Adjustment:     {indoor_eval['weather_adjustment']}")
print(f"  Recommendation: {indoor_eval['final_context_recommendation']}")
print(f"  Reason:         {indoor_eval['reason']}")



## 10. Strict Safety Priority Hierarchy

The system enforces a multi-tier safety precedence:
1. **Severe Water-Quality Safety (Sewer Bypass Cutoff)**
2. **Isolation Forest High-Risk Anomaly (`SEWER_BYPASS_OVERRIDE` / `HIGH_RISK_REVIEW`)**
3. **Supervised ML Classification (XGBoost Initial Route)**
4. **Environmental & Weather Context Layer (Phase 9)**
5. **Operational Storage Preferences (Phase 10)**

**Absolute Health Rule:**
If water quality demands `Sewer Bypass` or triggers a critical anomaly cutoff, sunny weather **CANNOT** permit irrigation.



In [ ]:
# Testing Safety Priority: Contaminated greywater with perfect sunny weather
sewer_eval = evaluate_weather_context(
    initial_route="Sewer Bypass",
    weather_data=dry_weather,
    water_quality_status="SEWER_BYPASS_OVERRIDE"
)

print("Evaluation of Sewer Bypass under Sunny Dry Weather:")
print(f"  Recommendation: {sewer_eval['final_context_recommendation']}")
print(f"  Adjustment:     {sewer_eval['weather_adjustment']}")
print(f"  Reason:         {sewer_eval['reason']}")



## 11. Systematic End-to-End Test Scenarios

We execute a comprehensive matrix of operational test cases across all routing classes and meteorological regimes:



In [ ]:
scenarios = [
    {
        "name": "Scenario 1: Nominal Restricted Irrigation in Clear Dry Weather",
        "route": "Restricted Irrigation",
        "weather": OFFLINE_DEMO_PROFILES["dry_clear"],
        "status": "NORMAL"
    },
    {
        "name": "Scenario 2: Restricted Irrigation during Heavy Rainstorm",
        "route": "Restricted Irrigation",
        "weather": OFFLINE_DEMO_PROFILES["heavy_rain"],
        "status": "NORMAL"
    },
    {
        "name": "Scenario 3: Contaminated Greywater (Sewer Bypass) on a Sunny Day",
        "route": "Sewer Bypass",
        "weather": OFFLINE_DEMO_PROFILES["dry_clear"],
        "status": "SEWER_BYPASS_OVERRIDE"
    },
    {
        "name": "Scenario 4: Indoor Reuse (Toilet Flushing) during Heavy Downpour",
        "route": "Indoor Reuse",
        "weather": OFFLINE_DEMO_PROFILES["heavy_rain"],
        "status": "NORMAL"
    },
    {
        "name": "Scenario 5: Restricted Irrigation when Weather Telemetry is Unavailable",
        "route": "Restricted Irrigation",
        "weather": {"weather_status": "UNAVAILABLE", "error_reason": "Gateway timeout"},
        "status": "NORMAL"
    }
]

results = []
for sc in scenarios:
    res = evaluate_weather_context(sc["route"], sc["weather"], sc["status"])
    results.append({
        "Scenario": sc["name"],
        "Initial Route": sc["route"],
        "Weather Status": res["weather_status"],
        "Context Recommendation": res["final_context_recommendation"],
        "Operational Adjustment": res["weather_adjustment"]
    })

scenario_summary_df = pd.DataFrame(results)
scenario_summary_df



## 12. Robust Error Handling & In-Memory Caching

To prevent API rate-limiting and buffer against intermittent network drops, queries are cached with a configurable Time-To-Live (`WEATHER_CACHE_TTL_SEC`).
In the event of network timeout or bad JSON responses, the provider gracefully reports `UNAVAILABLE` without crashing.



In [ ]:
cache = WeatherCache(default_ttl_sec=300)
cache.set("facility_weather", sample_weather)
cached = cache.get("facility_weather")
print(f"Cache retrieval successful: {cached['location']} at {cached['temperature_C']} °C")
print(f"Cache miss check for invalid key: {cache.get('non_existent_key')}")



## 13. System Validation Summary

All 13 weather context test cases in `tests/test_weather_context.py` and all 59 project-wide unit tests pass with 100% success rate:
- Valid weather response schema
- Invalid API response handling
- Network timeout resilience
- API unavailable fallback
- High rainfall suppression
- Low rainfall approval
- High precipitation probability pre-emption
- Indoor reuse hydraulic decoupling
- Irrigation deferral in rain
- Sewer bypass safety override immutability
- Missing weather field detection
- Invalid probability boundary detection
- Cache hit, miss, and expiration behavior



In [ ]:
# Summary visualization of decision rules
categories = ["Clear Dry", "Heavy Rain", "Freezing", "Torrential Storm"]
irrigation_outcomes = ["ALLOW_ROUTE", "STORE_FOR_LATER", "DEFER_ROUTE", "STORE_FOR_LATER"]
indoor_outcomes = ["ALLOW_ROUTE", "ALLOW_ROUTE", "ALLOW_ROUTE", "ALLOW_ROUTE"]
sewer_outcomes = ["SAFETY_OVERRIDE", "SAFETY_OVERRIDE", "SAFETY_OVERRIDE", "SAFETY_OVERRIDE"]

matrix_df = pd.DataFrame({
    "Condition": categories,
    "Restricted Irrigation": irrigation_outcomes,
    "Indoor Reuse": indoor_outcomes,
    "Sewer Bypass": sewer_outcomes
})
matrix_df.set_index("Condition", inplace=True)
print("Environmental Decision Matrix:")
matrix_df



## 14. Operational Limitations and Safety Disclaimer

> ### Important Operational Limitation
> **"Weather information provides operational context and does not independently establish water safety."**
>
> 1. **Sensor Grounding**: Meteorological predictions describe atmospheric conditions, not greywater biochemical composition. Low rainfall cannot purify high-BOD effluent.
> 2. **Forecast Uncertainty**: Precipitation probabilities represent stochastic estimates. Local micro-climates may diverge from regional meteorological forecasts.
> 3. **Runoff Vulnerability**: Soil saturation parameters depend on local soil percolation rates (hydraulic conductivity), which may vary across different sub-zones of an irrigation site.
> 4. **Storage Dependency**: When irrigation is suppressed due to weather, greywater must be temporarily routed to storage tanks, where microbial growth and biochemical decay (addressed in Phase 10) must be managed.

